In [1]:
import jax
import jax.numpy as jnp
import time
import argparse
import resource
from pathlib import Path

/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


In [2]:
import os
from stream import (
    run_stream
)

nbiter = 10
device = 'tpu'
device_name = 'TPUv5' if device == 'tpu' else 'cpu'

for dtype in ["float32"]:
    for size in [20000000, 40000000, 80000000, 160000000]:
        run_stream(size, nbiter, device, dtype)

        # Rename the resulting file to the requested format
        old_filename = f'stream_{dtype}.txt'
        new_filename = f'stream_{device_name}_{dtype}_N{size}.txt'
        if os.path.exists(old_filename):
            os.rename(old_filename, new_filename)
            print(f'Renamed {old_filename} to {new_filename}')

Renamed stream_float32.txt to stream_TPUv5_float32_N20000000.txt
Renamed stream_float32.txt to stream_TPUv5_float32_N40000000.txt
Renamed stream_float32.txt to stream_TPUv5_float32_N80000000.txt
Renamed stream_float32.txt to stream_TPUv5_float32_N160000000.txt


In [3]:
import zipfile
import os
from google.colab import files

# Define the name of the output zip file
zip_filename = 'stream_results.zip'

# Mapping of local directories to their desired names inside the zip
dump_mapping = {
    'jaxpr_dump': f'stream_{device_name}_{dtype}_jaxpr_dump',
    'hlo_dump': f'stream_{device_name}_{dtype}_hlo_dump'
}

# Create a zip archive
with zipfile.ZipFile(zip_filename, 'w') as zipf:
    # Add .txt files from the current directory
    txt_files = [f for f in os.listdir('.') if f.startswith('stream_') and f.endswith('.txt')]
    for file in txt_files:
        zipf.write(file)

    # Add files from dump folders into renamed subdirectories
    for local_folder, zip_folder in dump_mapping.items():
        if os.path.exists(local_folder):
            for root, dirs, files_in_dir in os.walk(local_folder):
                for file in files_in_dir:
                    file_path = os.path.join(root, file)
                    # Construct the internal path: new_folder_name/filename
                    archive_path = os.path.join(zip_folder, file)
                    zipf.write(file_path, arcname=archive_path)

# Download the zip file
files.download(zip_filename)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>